# Section 6 — Foundry Agent Service: Prompt Agents

## Storyline

So far every agent we've built runs **locally** — the `Agent` + `FoundryChatClient` pattern. The agent loop executes in our process; Foundry is just the model endpoint.

But Foundry also offers a **fully managed** path: **prompt agents**. You provide instructions, a model, and built-in tools — Foundry handles the agent loop, conversation history, and tool execution. No code, no containers.

```
  Your code (Sections 1–5)              Foundry Agent Service
  ────────────────────────              ─────────────────────
  Agent + FoundryChatClient             Prompt Agent
  You run the loop locally.             Foundry runs the loop.
  Full control.                         No code. Built-in tools.
```

## What you'll do

| Part | What | Why |
|------|------|-----|
| 1 | **Prompt agents** — create via SDK, add a built-in web search tool, chat | Understand the simplest Foundry agent type |
| 2 | **YAML agent definitions** — `agent.yaml` for prompt agents | Understand the declarative config for version control and CI/CD |

> **📌 What about Hosted Agents?** Foundry also supports **hosted agents** (currently in preview), where you bring your own code (e.g. a FastAPI app with custom logic) and Foundry hosts and scales it for you — no container management needed. Think of it as the middle ground between prompt agents (no code) and running your own container. This is out of scope for today's workshop, but worth exploring: [Hosted agents docs](https://learn.microsoft.com/en-us/azure/foundry/agents/overview).

References:
- [Foundry Agent Service overview](https://learn.microsoft.com/en-us/azure/foundry/agents/overview)
- [Microsoft Foundry quickstart](https://learn.microsoft.com/en-us/azure/foundry/quickstarts/get-started-code)

## Setup

> **Before you start:** In the cell below, change `USER_SUFFIX` to your name (lowercase, no spaces) — e.g. `"jan"`, `"sanne"`. This ensures your agents have unique names in the shared Foundry project.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
model = os.environ["FOUNDRY_MODEL"]

# ⚠️ Replace with your name (lowercase, no spaces) to avoid agent name collisions
USER_SUFFIX = "xxx"

print(f"Foundry endpoint: {endpoint}")
print(f"Model:            {model}")
print(f"User suffix:      {USER_SUFFIX}")

---
## Part 1 — Prompt Agents

A **prompt agent** is the simplest agent type in Foundry Agent Service. It's fully managed — you provide instructions, a model, and optionally tools. Foundry handles the agent loop, conversation history, and built-in tool execution (file search, code interpreter, web search).

**When to use:** rapid prototyping, simple Q&A with built-in tools, no infra to manage.

### Creating a prompt agent

You can create prompt agents two ways:
1. **Foundry portal** — point-and-click in the UI
2. **Programmatically** — via the `azure-ai-projects` SDK (what we'll do here)

Both use the same underlying API: `agents.create_version()` with a `PromptAgentDefinition`.

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential

credential = AzureCliCredential()

# Connect to the Foundry project
project_client = AIProjectClient(
    credential=credential,
    endpoint=endpoint,
)

print(f"Connected to project at: {endpoint}")

### Create a prompt agent with web search

We'll create a prompt agent with a **built-in web search tool**. This tool runs entirely server-side in Foundry — no function-calling loop, no client-side code. You can test it both in the notebook and in the Foundry portal playground.

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition, WebSearchTool

AGENT_NAME = f"policypal-prompt-{USER_SUFFIX}"

# Create a prompt agent with web search
agent = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=model,
        instructions=(
            "You are PolicyPal, an internal assistant for customer-service representatives "
            "at Athora Netherlands, a life insurance and pension company. "
            "Use web search to find current information when needed. "
            "Keep replies concise and factual."
        ),
        tools=[WebSearchTool()],  # built-in — runs server-side in Foundry
    ),
)

print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

### Chat with the prompt agent

To talk to a prompt agent you use the **Responses API** via `project_client.get_openai_client()`. The Foundry service manages conversation history — you pass `input` directly and get `output_text` back.

The web search tool runs **server-side** — Foundry executes the search and feeds the results to the model automatically. No function-calling loop needed on your side.

In [ ]:
# Get the OpenAI-compatible client for the Responses API
openai_client = project_client.get_openai_client()

# Create a server-side conversation in Foundry — history is managed by the service, not locally
conversation = openai_client.conversations.create()
print(f"Conversation created (id: {conversation.id})")

# Chat with the agent — web search runs server-side automatically
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": AGENT_NAME, "type": "agent_reference"}},
    input="What is Athora Netherlands and what products do they offer?",
)
print(f"Agent: {response.output_text}")

### How does this compare to what we've been doing?

| | Sections 1–5: `Agent` + `FoundryChatClient` | Prompt agent (this section) |
|---|---|---|
| **Agent loop** | Runs in your process | Runs in Foundry |
| **Conversation history** | You manage via `session` | Foundry manages via `conversation` |
| **Tools** | `@tool`-decorated Python functions (run locally) | Built-in tools (web search, file search, code interpreter) — run server-side |
| **Deployment** | You deploy your own container / run locally | Foundry manages everything |

> **Try it in the portal too!** Go to **Build > Agents** in the Foundry portal, click on your agent, and use the **Chat** tab to interact with it. The web search tool works there as well — Foundry handles everything server-side.

### A second turn

Let's ask a follow-up to see that Foundry manages conversation history across turns:

In [ ]:
# Follow-up question in the same conversation — Foundry remembers context
response2 = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": AGENT_NAME, "type": "agent_reference"}},
    input="What are the latest regulatory changes affecting them?",
)
print(f"Agent: {response2.output_text}")

---
## Part 2 — YAML Agent Definitions

Prompt agents can also be described declaratively in YAML via `agent.yaml`. This is important for:
- **Version control** — agent definitions live in git alongside your code
- **CI/CD** — deploy agents through pipelines, not portal clicks
- **Reproducibility** — anyone on the team can recreate the same agent from the YAML file

### Example: `agent.yaml` for a prompt agent

This YAML defines the same agent we just created via the SDK — model, instructions, and the web search tool. Everything the service needs to run it, with no code:

```yaml
kind: prompt
name: policypal-prompt-yaml
model: gpt-4.1
instructions: >
  You are PolicyPal, an internal assistant for customer-service representatives
  at Athora Netherlands, a life insurance and pension company.
  Use web search to find current information when needed.
  Keep replies concise and factual.
tools:
  - type: web_search
```

We already created this file as `agent.yaml` in the workspace. The script below appends your suffix to the agent name to avoid collisions:

In [ ]:
import yaml
from azure.ai.projects.models import PromptAgentDefinition, WebSearchTool

# Read the YAML file
with open("agent.yaml") as f:
    config = yaml.safe_load(f)

# Map YAML tool types to SDK tool objects
TOOL_MAP = {
    "web_search": WebSearchTool,
}

tools = [TOOL_MAP[t["type"]]() for t in config.get("tools", []) if t["type"] in TOOL_MAP]

# Append user suffix to avoid name collisions
yaml_agent_name = f"{config['name']}-{USER_SUFFIX}"

# Deploy the agent from YAML
agent_from_yaml = project_client.agents.create_version(
    agent_name=yaml_agent_name,
    definition=PromptAgentDefinition(
        model=config["model"],
        instructions=config["instructions"],
        tools=tools,
    ),
)

print(f"Agent deployed from YAML (id: {agent_from_yaml.id}, name: {agent_from_yaml.name}, version: {agent_from_yaml.version})")

### Clean up

Prompt agents persist in your Foundry project. Let's delete the one we created so we don't leave clutter behind.

In [ ]:
# Clean up the prompt agents we created
project_client.agents.delete(agent_name=AGENT_NAME)
print(f"Deleted {AGENT_NAME}")

project_client.agents.delete(agent_name=yaml_agent_name)
print(f"Deleted {yaml_agent_name}")

---
## Summary

### What you did

1. **Created a prompt agent** with a built-in web search tool via the SDK
2. **Chatted with it** using the Responses API — multi-turn, with Foundry managing conversation history
3. **Saw the YAML definition** that can version-control and CI/CD the same agent

### Prompt agents vs. Agent Framework (Sections 1–5)

| | Agent Framework (Sections 1–5) | Prompt Agent (this section) |
|---|---|---|
| **Agent loop** | Your code | Foundry-managed |
| **Tools** | `@tool` — your Python functions run locally | Built-in (web search, file search, code interpreter) — run server-side |
| **Conversation history** | You manage via `session` | Foundry manages via `conversation` |
| **Deployment** | You run locally or deploy a container | Foundry manages everything |
| **Best for** | Full control, complex logic, custom tools | Prototyping, simple agents, built-in tools |

### Next steps

- Try adding `file_search` or `code_interpreter` to the prompt agent
- Go to the [Foundry portal](https://ai.azure.com) and interact with `policypal-prompt` in the playground
- Read the [Foundry Agent Service overview](https://learn.microsoft.com/en-us/azure/foundry/agents/overview) for more information